# ParPTE · Writing trait model (ELLIPSE)

Fine-tunes DeBERTa-v3 to rate learner essays on six analytic traits — **cohesion, syntax, vocabulary, phraseology, grammar, conventions** (1.0–5.0) — and exports it to ONNX for fast CPU inference.
ParPTE maps them onto PTE traits:

| ELLIPSE | PTE trait |
|---|---|
| grammar + conventions | Grammar |
| vocabulary + phraseology | Vocabulary (range) |
| cohesion | Development, Structure & Coherence |
| syntax + vocabulary | General Linguistic Range |

**Data:** the ELLIPSE corpus from the *Feedback Prize – English Language Learning* competition (≈3,900 essays by English learners, double-rated).
Click **Add Input → Competitions → feedback-prize-english-language-learning**. You must join the competition and accept its rules first.

**Licence check before commercial use:** open the competition's *Data* tab and read the licence. If it doesn't allow commercial use, keep this model for your own practice and let paying users rely on the rule-based writing scorer, which needs no trained model.

**Settings:** Accelerator **GPU T4 x1** or **P100**, Internet **On**. Takes ≈2 h for `deberta-v3-base` (≈40 min for `deberta-v3-small`).

In [ ]:
import os, json, math, random, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "sentencepiece"], check=True)
import numpy as np, pandas as pd, torch, torch.nn as nn
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from tqdm.auto import tqdm

BASE_MODEL = "microsoft/deberta-v3-base"   # "microsoft/deberta-v3-small" for a faster, smaller model
MAX_LEN, EPOCHS, BS, LR, SEED = 512, 4, 8, 2e-5, 42
TARGETS = ["cohesion", "syntax", "vocabulary", "phraseology", "grammar", "conventions"]
OUT = "/kaggle/working/parpte_models/essay_traits"
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda"

path = next(p for p in ["/kaggle/input/feedback-prize-english-language-learning/train.csv"] if os.path.exists(p))
df = pd.read_csv(path)
val = df.sample(frac=0.1, random_state=SEED)
train = df.drop(val.index)
print(len(train), "train /", len(val), "validation essays")

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

class Essays(torch.utils.data.Dataset):
    def __init__(self, d): self.t, self.y = d.full_text.tolist(), d[TARGETS].values.astype(np.float32)
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return self.t[i], self.y[i]

def collate(b):
    enc = tok([x for x, _ in b], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    return enc, torch.tensor(np.stack([y for _, y in b]))

class TraitModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(BASE_MODEL)
        self.head = nn.Linear(self.enc.config.hidden_size, len(TARGETS))
    def forward(self, input_ids, attention_mask):
        h = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        return self.head(pooled) + 3.0   # centre outputs on the 1–5 scale

model = TraitModel().to(device)
dl_tr = torch.utils.data.DataLoader(Essays(train), batch_size=BS, shuffle=True, collate_fn=collate, num_workers=2)
dl_va = torch.utils.data.DataLoader(Essays(val), batch_size=16, collate_fn=collate)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = get_cosine_schedule_with_warmup(opt, int(0.1 * len(dl_tr) * EPOCHS), len(dl_tr) * EPOCHS)
scaler = torch.cuda.amp.GradScaler()

def evaluate():
    model.eval(); P, Y = [], []
    with torch.no_grad():
        for enc, y in dl_va:
            with torch.autocast("cuda", dtype=torch.float16):
                P.append(model(enc["input_ids"].to(device), enc["attention_mask"].to(device)).float().cpu().numpy())
            Y.append(y.numpy())
    P, Y = np.concatenate(P), np.concatenate(Y)
    rmse = np.sqrt(((P - Y) ** 2).mean(0))
    return float(rmse.mean()), dict(zip(TARGETS, rmse.round(3).tolist()))

best = 9.9
for ep in range(EPOCHS):
    model.train()
    for enc, y in tqdm(dl_tr, desc=f"epoch {ep+1}"):
        with torch.autocast("cuda", dtype=torch.float16):
            pred = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
            loss = nn.functional.smooth_l1_loss(pred.float(), y.to(device), beta=0.5)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
    mcrmse, per = evaluate()
    print(f"epoch {ep+1}: MCRMSE {mcrmse:.4f}", per)   # competition winners ≈0.43; a solid single model ≈0.45–0.46
    if mcrmse < best:
        best = mcrmse
        torch.save(model.state_dict(), "/kaggle/working/best.pt")

## Export to ONNX (CPU inference in the ParPTE API)

In [ ]:
import onnxruntime as ort
model.load_state_dict(torch.load("/kaggle/working/best.pt"))
model = model.float().cpu().eval()
sample = tok(val.full_text.iloc[0], truncation=True, max_length=MAX_LEN, return_tensors="pt")
torch.onnx.export(model, (sample["input_ids"], sample["attention_mask"]), f"{OUT}/model.onnx",
                  input_names=["input_ids", "attention_mask"], output_names=["traits"],
                  dynamic_axes={"input_ids": {0: "b", 1: "t"}, "attention_mask": {0: "b", 1: "t"}, "traits": {0: "b"}},
                  opset_version=17, dynamo=False)
tok.backend_tokenizer.save(f"{OUT}/tokenizer.json")
json.dump({"base_model": BASE_MODEL, "max_len": MAX_LEN, "targets": TARGETS, "val_mcrmse": best}, open(f"{OUT}/config.json", "w"), indent=2)

# sanity check: ONNX + the standalone tokenizer (exactly what the server uses) match PyTorch
from tokenizers import Tokenizer
t2 = Tokenizer.from_file(f"{OUT}/tokenizer.json"); t2.enable_truncation(MAX_LEN)
e = t2.encode(val.full_text.iloc[0])
sess = ort.InferenceSession(f"{OUT}/model.onnx")
onnx_out = sess.run(None, {"input_ids": np.array([e.ids]), "attention_mask": np.array([e.attention_mask])})[0]
with torch.no_grad():
    torch_out = model(sample["input_ids"], sample["attention_mask"]).numpy()
print("onnx ", onnx_out.round(2)); print("torch", torch_out.round(2))
assert np.allclose(onnx_out, torch_out, atol=0.05), "ONNX output differs from PyTorch"

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/parpte_writing_models", "zip", "/kaggle/working/parpte_models")
print("Download parpte_writing_models.zip from the Output panel and unzip it into backend/models/ (it contains essay_traits/)")